# Phase 6B — Convolutional Neural Networks (CNNs)

**Theory:** Convolution operation, filters/kernels, feature maps, pooling, receptive field, transfer learning.

**Install:** `pip install torch torchvision`

---

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader
    import torchvision
    import torchvision.transforms as transforms
    from torchvision import models

    TORCH_AVAILABLE = True
    print(f"PyTorch: {torch.__version__}, torchvision: {torchvision.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("Install: pip install torch torchvision")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

---
## 1. How Convolution Works

A convolution slides a **filter (kernel)** over an image, computing dot products at each position. Different filters detect different features (edges, textures, shapes).

In [ ]:
import numpy as np
from scipy.signal import convolve2d

# Create a simple test image (checkerboard)
size = 8
image = np.zeros((size, size))
for i in range(size):
    for j in range(size):
        if (i + j) % 2 == 0:
            image[i, j] = 1.0

# Common edge-detecting filters
filters = {
    "Vertical Edge\n(Sobel X)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Horizontal Edge\n(Sobel Y)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Sharpening": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Gaussian Blur": np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16,
}

fig, axes = plt.subplots(1, len(filters) + 1, figsize=(16, 3.5))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original Image")
axes[0].axis("off")

for ax, (name, kernel) in zip(axes[1:], filters.items()):
    feature_map = convolve2d(image, kernel, mode="same")
    ax.imshow(feature_map, cmap="gray")
    ax.set_title(f"{name}\nkernel={kernel.shape}")
    ax.axis("off")

plt.suptitle(
    "Convolution: Same Image, Different Filters → Different Feature Maps", fontsize=12
)
plt.tight_layout()
plt.show()

---
## 2. Building a CNN from Scratch

In [ ]:
if TORCH_AVAILABLE:

    class SimpleCNN(nn.Module):
        """
        CNN for MNIST (28x28 grayscale, 10 classes).
        Architecture:
          Conv1 (1→32, 3x3) → ReLU → MaxPool(2x2)
          Conv2 (32→64, 3x3) → ReLU → MaxPool(2x2)
          Flatten → FC(1600→128) → Dropout → FC(128→10)
        """

        def __init__(self):
            super().__init__()
            self.conv_block1 = nn.Sequential(
                nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 → 14x14
            )
            self.conv_block2 = nn.Sequential(
                nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 → 7x7
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(64 * 7 * 7, 128),
                nn.ReLU(),
                nn.Dropout(p=0.5),
                nn.Linear(128, 10),
            )

        def forward(self, x):
            x = self.conv_block1(x)
            x = self.conv_block2(x)
            return self.classifier(x)

    model = SimpleCNN()
    print(model)

    # Test with a dummy batch
    dummy = torch.randn(4, 1, 28, 28)  # batch of 4 grayscale 28x28 images
    out = model(dummy)
    print(f"\nInput shape:  {dummy.shape}")
    print(f"Output shape: {out.shape}   (4 images, 10 class logits)")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

---
## 3. Transfer Learning with Pretrained Models

**Transfer learning:** Take a model pretrained on a large dataset (ImageNet — 1.2M images, 1000 classes) and fine-tune it for your task.

**Why it works:** Early layers learn universal features (edges, textures) reusable across tasks.

In [ ]:
if TORCH_AVAILABLE:
    # Load pretrained ResNet18 (11M parameters, trained on ImageNet)
    resnet = models.resnet18(weights="DEFAULT")
    print(
        f"ResNet18 loaded — {sum(p.numel() for p in resnet.parameters()):,} parameters"
    )

    # Strategy 1: Feature extraction — freeze all layers, replace last layer only
    for param in resnet.parameters():
        param.requires_grad = False  # freeze all

    n_classes = 5  # your custom task
    resnet.fc = nn.Linear(512, n_classes)  # replace final layer (unfrozen by default)

    trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    total = sum(p.numel() for p in resnet.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,}  ({trainable / total:.1%})")

    # Strategy 2: Fine-tuning — unfreeze later layers too
    # for param in resnet.layer4.parameters():
    #     param.requires_grad = True

    # Standard ImageNet preprocessing
    transform = transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    print("\nTransform pipeline ready for ImageNet-pretrained models")

---
## 4. CNN Architecture Summary

```
Input Image
    ↓
[Conv → BatchNorm → ReLU → Pool] × N   ← Feature Extraction
    ↓
Flatten
    ↓
[Linear → ReLU → Dropout] × M           ← Classification Head
    ↓
Output (softmax / sigmoid)
```

**Popular architectures:**
- **ResNet** (2015) — skip connections prevent vanishing gradients; ResNet18/50/101
- **VGG** (2014) — simple stacked convolutions; easy to understand
- **EfficientNet** (2019) — state-of-the-art efficiency
- **ViT** (2020) — Vision Transformer; attention-based, no convolutions

**Key hyperparameters:**
- `kernel_size` — filter size (3x3 most common)
- `padding=1` with `kernel_size=3` → same spatial size output
- `MaxPool2d(2,2)` — halves spatial dimensions
- `in_channels` / `out_channels` — number of input/output feature maps